# 당뇨 예측 - LightGBM 베이스 모델

- 타겟: `당뇨유병` (0: 없음 / 1: 있음)
- 모델: LightGBM Classifier
- 평가지표: Recall (주), F1-score, AUC-ROC

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# ── 경로 설정 ──────────────────────────────────────────────
INPUT_PATH   = '/Users/Jiyeon/Desktop/final_project/ML/data/x1_preprocessed.csv'
RANDOM_STATE = 42

## 1. 데이터 로드

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f'로드 완료 | shape: {df.shape}')
print(f'컬럼: {df.columns.tolist()}')

## 2. 피처 / 타겟 분리 & 클래스 불균형 확인

In [ ]:
TARGET    = '당뇨유병'
DROP_COLS = ['고혈압유병', '당뇨유병', '이상지질혈증유병', '비만단계']

data = df.dropna(subset=[TARGET]).copy()
X = data.drop(columns=DROP_COLS)
y = data[TARGET].astype(int)

print(f'샘플 수: {len(y)}')
print(f'클래스 분포:\n{y.value_counts()}')
print(f'불균형 비율 (0:1): {y.value_counts()[0]}:{y.value_counts()[1]}')

classes = np.array([0, 1])
cw = compute_class_weight('balanced', classes=classes, y=y)
scale_pos_weight = cw[1] / cw[0]
print(f'\nscale_pos_weight: {scale_pos_weight:.4f}')

## 3. Train / Test 분리

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train 양성 비율: {y_train.mean():.3f}')
print(f'Test  양성 비율: {y_test.mean():.3f}')

## 4. 베이스 모델 학습

In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

model.fit(X_train, y_train)
print('학습 완료')

## 5. 평가

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

recall  = recall_score(y_test, y_pred)
f1      = f1_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_proba)

print('=' * 45)
print('  [당뇨 LightGBM 베이스 모델 결과]')
print('=' * 45)
print(f'  Recall   : {recall:.4f}')
print(f'  F1-score : {f1:.4f}')
print(f'  AUC-ROC  : {auc_roc:.4f}')
print('=' * 45)
print()
print('[분류 리포트]')
print(classification_report(y_test, y_pred, target_names=['정상(0)', '당뇨(1)']))

## 6. 혼동 행렬

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['정상(0)', '당뇨(1)'])
disp.plot(cmap='Greens')
plt.title('Confusion Matrix - 당뇨 LightGBM')
plt.tight_layout()
plt.show()

## 7. ROC 커브

In [ ]:
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay.from_predictions(y_test, y_proba, name='LightGBM')
plt.title('ROC Curve - 당뇨 LightGBM')
plt.tight_layout()
plt.show()

## 8. Feature Importance (Top 20)

In [ ]:
fi = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
fi_top20 = fi.head(20)

plt.figure(figsize=(8, 6))
fi_top20[::-1].plot(kind='barh', color='seagreen')
plt.title('Feature Importance Top 20 - 당뇨 LightGBM')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(fi_top20)

## 9. Stratified K-Fold 교차 검증 (5-fold)

In [ ]:
cv_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    cv_model, X, y, cv=cv,
    scoring=['recall', 'f1', 'roc_auc'],
    n_jobs=-1,
)

print('=== 5-Fold CV 결과 ===')
for metric in ['recall', 'f1', 'roc_auc']:
    scores = cv_results[f'test_{metric}']
    print(f'  {metric:8s}: {scores.mean():.4f} ± {scores.std():.4f}')